# Nemotron Reasoning LoRA

Self-contained training notebook: single-epoch LoRA fine-tune of
**Nemotron-3-Nano-30B-A3B** (hybrid Mamba-2 + MoE), producing a LoRA adapter
packaged as `submission.zip`.

All code is inline so the notebook runs standalone on Kaggle. The same logic is
also available as an importable package under `src/nemotron_lora/` in the repo.

Key techniques: LoRA factors tied across all 128 MoE experts, Cut Cross-Entropy
fused `lm_head` + loss (keeps the 30B model in VRAM), manual `lm_head` LoRA with
key-prefix fix, Mamba CUDA fast path, cosine LR `2e-4 -> 2e-5` with mild dropout
and weight decay over a single epoch.

In [ ]:
# Offline package install (Kaggle layout). Adjust paths for your environment.
import os
import subprocess

PKG = "/kaggle/input/datasets/mayukh18/nemotron-packages"
subprocess.run(
    f"pip install -q --no-index --find-links {PKG}/packages "
    "unsloth trl peft transformers datasets accelerate bitsandbytes",
    shell=True, check=True,
)
for whl in ("causal_conv1d", "mamba_ssm"):
    subprocess.run(f"pip install -q {PKG}/{whl}-*.whl", shell=True, check=True)

RTX = "/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"
if os.path.isdir(RTX):
    subprocess.run(
        ["pip", "install", "-q", "--no-index", "--find-links", RTX,
         "protobuf==6.33.5", "sentencepiece", "safetensors", "huggingface_hub"],
        check=False,
    )
print("installs done")

In [ ]:
# Configuration ---------------------------------------------------------------
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "up_proj", "down_proj",
    "in_proj", "out_proj",
    "lm_head",
]
MOE_TIE_WEIGHTS = True  # tie the LoRA factors across all 128 MoE experts

MAX_SEQ_LEN = 8192
NUM_STEPS = 1000        # clamped to one epoch over the corpus
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 4
LEARNING_RATE = 2e-4
LR_END = 2e-5           # cosine decay target
WEIGHT_DECAY = 0.01
SEED = 42

# Data + model locations (Kaggle layout). Point REPO_BASE at the extracted
# corpus snapshot; the base model is pulled via kagglehub.
REPO_BASE = "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master"
CORPUS_PATH = f"{REPO_BASE}/training/sft/04-08-16-14/tokens"
TRAIN_ORDER_PATH = f"{REPO_BASE}/training/sft/04-08-16-14/logprobs/index.jsonl"
MODEL_HANDLE = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"

In [ ]:
# Data loading ----------------------------------------------------------------
import json
import os


def load_training_order(train_order_path):
    """De-duplicated problem ids for the first epoch, in file order."""
    ordered_ids, seen = [], set()
    with open(train_order_path) as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("epoch", 0) != 0:
                continue
            pid = rec["problem_id"]
            if pid in seen:
                continue
            seen.add(pid)
            ordered_ids.append(pid)
    return ordered_ids


def load_examples(corpus_path, ordered_ids, max_seq_len):
    """Next-token examples: tokens (inputs), targets (shift +1), weights (mask)."""
    examples = []
    for sid in ordered_ids:
        seg_path = os.path.join(corpus_path, sid, "synthetic.json")
        assert os.path.isfile(seg_path), f"missing corpus segment for {sid}"
        with open(seg_path) as f:
            rec = json.load(f)
        tokens, mask = rec["tokens"], rec["mask"]
        if not tokens:
            continue
        if len(tokens) > max_seq_len:
            tokens, mask = tokens[:max_seq_len], mask[:max_seq_len]
        if not any(mask):
            continue
        examples.append({
            "problem_id": sid,
            "tokens": tokens[:-1],
            "targets": tokens[1:],
            "weights": [float(m) for m in mask[1:]],
        })
    return examples

In [ ]:
# Training --------------------------------------------------------------------
import gc
import math
import random
import sys
import time
import zipfile

import kagglehub
import torch
from unsloth import FastLanguageModel
from cut_cross_entropy import linear_cross_entropy
from peft import LoraConfig
from peft.tuners.lora import Linear as LoraLinear
from safetensors.torch import load_file, save_file


def build_model(model_path):
    model, _tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    FastLanguageModel.for_training(model)

    # Enable the Mamba CUDA fast path (off by default under training).
    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
            nemotron_mod = _m
            break
    assert nemotron_mod is not None, "modeling_nemotron_h not found"
    nemotron_mod.is_fast_path_available = True

    # Unsloth skips lm_head LoRA for MoE models; add it back by hand.
    causal_lm = model
    while hasattr(causal_lm, "model"):
        causal_lm = causal_lm.model
    lm_head = causal_lm.lm_head
    if not isinstance(lm_head, LoraLinear):
        lcfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            lcfg, "default", target=lm_head, target_name="lm_head", parent=causal_lm,
        )

    # LoRA factors in fp32; base weights bf16 (MoE router already fp32).
    for name, param in model.named_parameters():
        if ".lora_" in name:
            param.data = param.data.to(torch.float32)
    for name, param in model.named_parameters():
        if ".lora_" in name:
            assert param.dtype == torch.float32, f"LoRA {name} expected fp32"
        elif ".mixer.gate." in name:
            assert param.dtype == torch.float32, f"router {name} expected fp32"
        else:
            assert param.dtype == torch.bfloat16, f"{name} expected bf16"
    return model


def patch_forward_cce(model):
    """Cut Cross-Entropy forward: fuse lm_head + loss, cache per-token CE."""
    base = model
    while hasattr(base, "model"):
        base = base.model

    def _forward(input_ids=None, attention_mask=None, labels=None, **kwargs):
        backbone_out = base.backbone(
            input_ids=input_ids, attention_mask=attention_mask,
            **{k: v for k, v in kwargs.items() if k in ("position_ids", "past_key_values", "use_cache")},
        )
        hidden_states = backbone_out[0]
        lm_head = base.lm_head
        lm_weight = lm_head.base_layer.weight + lm_head.scaling["default"] * (
            lm_head.lora_B["default"].weight @ lm_head.lora_A["default"].weight
        )
        if labels is not None:
            model._cached_per_token_ce = linear_cross_entropy(
                hidden_states, lm_weight, labels, reduction="none",
            )
            return model._cached_per_token_ce.mean()
        model._cached_per_token_ce = None
        return None

    base.forward = _forward


def moe_tied_params(model):
    """MoE expert LoRA factors that should share one update."""
    w1_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
    w2_names = ("down_proj", ".w2.")
    tied = []
    for name, param in model.named_parameters():
        if not param.requires_grad or ".experts." not in name or ".lora_" not in name:
            continue
        is_w1 = any(p in name for p in w1_names)
        is_w2 = any(p in name for p in w2_names)
        should_tie = (is_w1 and ".lora_A." in name) or (is_w2 and ".lora_B." in name)
        if not should_tie or param.dim() < 2 or param.shape[0] <= 1:
            continue
        tied.append(param)
    return tied


def save_adapter(model, output_dir="."):
    """Save adapter, fix the lm_head key prefix, zip for submission."""
    for f in os.listdir(output_dir):
        if f.startswith("adapter"):
            os.remove(os.path.join(output_dir, f))
    model.save_pretrained(output_dir)

    st_path = os.path.join(output_dir, "adapter_model.safetensors")
    tensors = load_file(st_path)
    save_file(
        {k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v
         for k, v in tensors.items()},
        st_path,
    )

    adapter_files = [f for f in os.listdir(output_dir) if f.startswith("adapter")]
    zip_path = os.path.join(output_dir, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in adapter_files:
            zf.write(os.path.join(output_dir, fname), fname)
    for fname in adapter_files:
        os.remove(os.path.join(output_dir, fname))
    print(f"Wrote {zip_path}")
    return zip_path


def train():
    random.seed(SEED)
    model_path = kagglehub.model_download(MODEL_HANDLE)

    ordered_ids = load_training_order(TRAIN_ORDER_PATH)
    examples = load_examples(CORPUS_PATH, ordered_ids, MAX_SEQ_LEN)
    random.Random(0).shuffle(examples)
    total_tokens = sum(len(e["tokens"]) for e in examples)
    print(f"{len(examples)} examples, {total_tokens:,} tokens")

    gc.collect()
    torch.cuda.empty_cache()
    model = build_model(model_path)
    patch_forward_cce(model)

    tied = moe_tied_params(model) if MOE_TIE_WEIGHTS else []
    if tied:
        with torch.no_grad():
            for p in tied:
                p.data.copy_(p.data.mean(dim=0, keepdim=True).expand_as(p.data))
    print(f"MoE tied params: {len(tied)}")

    def tie_grads():
        with torch.no_grad():
            for p in tied:
                if p.grad is not None:
                    p.grad.copy_(p.grad.sum(dim=0, keepdim=True).expand_as(p.grad))

    device = next(model.parameters()).device
    num_steps = min(NUM_STEPS, len(examples) // BATCH_SIZE)
    print(f"Training {num_steps} steps (batch={BATCH_SIZE}, micro={MICRO_BATCH_SIZE})")

    optimizer = None
    step = 0
    for batch_start in range(0, len(examples), BATCH_SIZE):
        if step >= num_steps:
            break
        batch = examples[batch_start: batch_start + BATCH_SIZE]
        n = len(batch)
        n_accum = math.ceil(n / MICRO_BATCH_SIZE)

        for mb_start in range(0, n, MICRO_BATCH_SIZE):
            mb = batch[mb_start: mb_start + MICRO_BATCH_SIZE]
            n_micro = len(mb)
            max_len = max(len(e["tokens"]) for e in mb)

            inp = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            tgt = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            wts = torch.zeros(n_micro, max_len, dtype=torch.float32, device=device)
            attn = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
            for i, e in enumerate(mb):
                sl = len(e["tokens"])
                inp[i, :sl] = torch.tensor(e["tokens"], dtype=torch.long)
                tgt[i, :sl] = torch.tensor(e["targets"], dtype=torch.long)
                wts[i, :sl] = torch.tensor(e["weights"], dtype=torch.float32)
                attn[i, :sl] = 1

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                model(input_ids=inp, attention_mask=attn, labels=tgt, use_cache=False)
                per_token_ce = model._cached_per_token_ce
                weight_sum = wts.sum()
                loss = (per_token_ce * wts).sum() / weight_sum if weight_sum > 0 else per_token_ce.sum() * 0.0
            (loss / n_accum).backward()
            del loss, per_token_ce

        if optimizer is None:
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=LEARNING_RATE, betas=(0.9, 0.95), eps=1e-8, weight_decay=WEIGHT_DECAY,
            )
        progress = step / max(num_steps - 1, 1)
        lr = LR_END + (LEARNING_RATE - LR_END) * 0.5 * (1 + math.cos(math.pi * progress))
        for pg in optimizer.param_groups:
            pg["lr"] = lr
        tie_grads()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1e9,
        )
        optimizer.step()
        optimizer.zero_grad()
        step += 1
        print(f"  step {step}/{num_steps}: grad_norm={grad_norm:.4f}, lr={lr:.2e}", flush=True)

    print(f"Done. Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    return save_adapter(model)

In [ ]:
train()  # writes submission.zip